# ImamAli110 — Cinematic GPU Production (Google Colab)
این دفترچه نسخهٔ تولیدی مسیر Colab است: اسکریپت فارسی → گویندگی فارسی → صحنه‌های سینمایی که بر اساس معنای روایت انتخاب می‌شوند → تدوین عمودی 9:16 → بارگذاری خصوصی در YouTube.

**GPU واقعی الزامی است.** این notebook عمداً روی CPU ادامه نمی‌دهد. ویدئو تا وقتی خودت در YouTube آن را بررسی نکنی روی `private` می‌ماند.

## 1. فعال‌کردن GPU
در Colab از منو برو به **Runtime → Change runtime type → T4 GPU** (اگر برای حسابت موجود است)، سپس `Run all` را اجرا کن. سهمیهٔ رایگان GPU تضمین‌شده یا نامحدود نیست.

In [ ]:
import os, subprocess, sys, torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU فعال نیست. در Runtime → Change runtime type یک GPU انتخاب کن و دوباره اجرا کن؛ تولید CPU عمداً غیرفعال است.')
print('GPU:', torch.cuda.get_device_name(0))
assert torch.cuda.get_device_capability(0)[0] >= 7, 'این GPU برای مسیر پیشنهادی مناسب نیست.'

In [ ]:
# دریافت نسخهٔ تازهٔ مخزن اصلی
REPO='/content/Youtube1'
if not os.path.isdir(REPO + '/.git'):
    subprocess.run(['git','clone','--depth','1','--branch','main','https://github.com/farzin0369/Youtube1.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'fetch','origin','main'],check=True)
    subprocess.run(['git','-C',REPO,'checkout','main'],check=True)
    subprocess.run(['git','-C',REPO,'reset','--hard','origin/main'],check=True)
os.chdir(REPO)
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','ffmpeg','fonts-dejavu-core','fonts-noto-core'],check=True)
# Keep Colab's CUDA-enabled PyTorch; do not replace it with a CPU-only wheel.
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt','piper-tts','diffusers>=0.32,<0.42','transformers>=4.46','accelerate>=1.0','safetensors','imageio-ffmpeg'],check=True)
subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
print('Repository and video dependencies installed.')

## 2. کلیدهای امن
در Colab پنل **Secrets** (آیکون کلید) را باز کن و در صورت نیاز این نام‌ها را اضافه کن: `YOUTUBE_CLIENT_ID`, `YOUTUBE_CLIENT_SECRET`, `YOUTUBE_REFRESH_TOKEN`. برای نوشتن اسکریپت بهتر می‌توانی `QWEN_API_KEY` و در صورت استفاده `QWEN_API_BASE` و `QWEN_MODEL` را هم اضافه کنی. گزینهٔ Notebook access را برای این notebook روشن کن.

کلیدها را داخل GitHub، notebook یا پیام چت ننویس. اگر توکن YouTube قبلی همان خطای `invalid_grant` بدهد، باید refresh token معتبر جدید در Secrets قرار بگیرد.

In [ ]:
from google.colab import userdata
def read_secret(name, default=''):
    try:
        value = userdata.get(name)
        return value.strip() if value else default
    except Exception:
        return default

for name in ['YOUTUBE_CLIENT_ID','YOUTUBE_CLIENT_SECRET','YOUTUBE_REFRESH_TOKEN','QWEN_API_KEY','QWEN_API_BASE','QWEN_MODEL']:
    value = read_secret(name)
    if value:
        os.environ[name] = value
    elif name.startswith('YOUTUBE_'):
        print('Missing Colab Secret:', name)

if not all(os.environ.get(k) for k in ['YOUTUBE_CLIENT_ID','YOUTUBE_CLIENT_SECRET','YOUTUBE_REFRESH_TOKEN']):
    raise RuntimeError('سه Secret مربوط به YouTube را در پنل Colab اضافه کن؛ بدون آن‌ها انتشار انجام نمی‌شود.')
os.environ['AI_ENGINE'] = 'qwen' if os.environ.get('QWEN_API_KEY') else 'local'
os.environ['VIDEO_ENGINE'] = 'cogvideox'
os.environ['LOCAL_VIDEO_MODEL'] = 'THUDM/CogVideoX-2b'
os.environ['TTS_PROVIDER'] = 'edge'

os.environ['PIPER_MODEL'] = '/content/fa_IR-amir-medium.onnx'
os.environ['TTS_VOICE'] = 'fa-IR-FaridNeural'

os.environ['VIDEO_CLIPS_SHORT'] = '6'
os.environ['VIDEO_STEPS'] = '25'
os.environ['VIDEO_FRAMES'] = '49'
os.environ['VIDEO_GUIDANCE'] = '6'
os.environ['HF_HOME'] = '/content/hf-cache'
print('Secrets loaded without displaying their values.')

In [ ]:
# Download free Persian Piper voice
import urllib.request
for url, dest in [
 ('https://huggingface.co/rhasspy/piper-voices/resolve/main/fa/fa_IR/amir/medium/fa_IR-amir-medium.onnx','/content/fa_IR-amir-medium.onnx'),
 ('https://huggingface.co/rhasspy/piper-voices/resolve/main/fa/fa_IR/amir/medium/fa_IR-amir-medium.onnx.json','/content/fa_IR-amir-medium.onnx.json')]:
    if not os.path.exists(dest): urllib.request.urlretrieve(url,dest)
print('Persian voice ready.')

## 3. تولید و بارگذاری آزمایشی
پیش‌فرض ویدئو `short` و وضعیت بارگذاری `private` است. شش صحنهٔ متحرک با پرامپت‌های سینمایی ساخته می‌شوند؛ انتخاب صحنه‌ها بر اساس موضوع گفتار مثل مهربانی، صبر، شکرگزاری، صداقت یا بخشش انجام می‌شود. این روش هم‌ترازی معنایی را بهتر می‌کند، اما تضمین نمی‌کند هر فریم دقیقاً با هر کلمه هم‌زمان شود.

In [ ]:
KIND = 'short'  # change to 'long' only when you have enough GPU time
cmd = [sys.executable, '-m', 'app.pipeline', '--kind', KIND, '--publish-mode', 'private']
print('Starting GPU video generation. The upload remains PRIVATE.')
subprocess.run(cmd, check=True, env=os.environ.copy())
print('Pipeline finished. Review the audit and video in output/.')

In [ ]:
from pathlib import Path
import zipfile
out = Path('/content/Youtube1/output')
files = list(out.rglob('*')) if out.exists() else []
for p in files:
    if p.is_file() and p.suffix.lower() in {'.mp4','.srt','.txt','.json','.jpg'}:
        print(p.relative_to(out), f'{p.stat().st_size/1024/1024:.2f} MB')
archive='/content/ImamAli110-output.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in files:
        if p.is_file(): z.write(p, p.relative_to(out))
print('Output archive:', archive)
from google.colab import files
files.download(archive)

## نکتهٔ اجرایی
این notebook از GitHub آخرین کد را می‌گیرد، اما **GitHub Actions به‌تنهایی نمی‌تواند نشست رایگان Colab را به‌طور قابل‌اعتماد و دائمی بیدار نگه دارد یا اجرای خودکار روزانه را تضمین کند**. این notebook مسیر GPU واقعی را فراهم می‌کند؛ پس از اجرای موفق باید log و وضعیت ویدئوی خصوصی را بررسی کرد. اگر OAuth خطای `invalid_grant` بدهد، مشکل از اعتبار توکن است نه کیفیت تولید.